# Validación automática del conjunto candidato

Este notebook ejecuta las siete comprobaciones de calidad exigidas sobre el CSV candidato. Todas las reglas provienen de `src/validadores.py`, la misma implementación utilizada por `pytest`, y cada fallo incluye cantidad, explicación y ejemplos suficientes para corregirlo.

## Configuración y carga reproducible

In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data" / "raw" / "establecimientos_raw.csv").exists():
    raise FileNotFoundError("Ejecute el notebook desde la raíz del repositorio o desde notebooks/.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 40)

from src.validadores import (
    cargar_csv_para_validacion,
    ejecutar_validaciones,
    resumen_validaciones,
)
from src.metricas_calidad import (
    calcular_comparacion_calidad,
    resumen_decisiones_duplicados,
    tabla_comparativa,
)

In [2]:
RUTA_CANDIDATO = ROOT / "data" / "processed" / "establecimientos_limpios_candidato.csv"
df_candidato = cargar_csv_para_validacion(RUTA_CANDIDATO)

print(f"Registros: {df_candidato.shape[0]:,}")
print(f"Variables: {df_candidato.shape[1]}")
df_candidato.head()

Registros: 11,868
Variables: 18


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ZONA_CAPITAL,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,15-01-0050-46,15-01-0890,BAJA VERAPAZ,SALAMA,<NA>,ESCUELA NORMAL RURAL NO.4 DR. ELIZARDO URIZAR ...,13 AVENIDA 7-41 ZONA 2 BARRIO HACIENDA DE LA V...,79400455,IRIS QUETZALI ORDOÑEZ TURCIOS,ANA ISABEL GUEVARA MEZA,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),BAJA VERAPAZ
1,15-01-0051-46,15-01-0890,BAJA VERAPAZ,SALAMA,<NA>,COLEGIO PARTICULAR MIXTO TEZULUTLAN,"DIAGONAL 4, 1-24 ZONA 2",79400182,IRIS QUETZALI ORDOÑEZ TURCIOS,ELIA NÍVEA LÓPEZ GARCÍA,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
2,15-01-0052-46,15-01-0895,BAJA VERAPAZ,SALAMA,<NA>,LICEO MIXTO SAN MATEO,9A. AVENIDA 6-67 ZONA 1,79401926,NELSON JOEL HERNANDEZ HERNANDEZ,CARMEN ALICIA HERNÁNDEZ GUILLERMO,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
3,15-01-0087-46,15-01-0890,BAJA VERAPAZ,SALAMA,<NA>,INSTITUTO NACIONAL DE EDUCACION DIVERSIFICADA,"13 AVENIDA 7-41, ZONA 2, BARRIO HACIENDA DE LA...",53009229,IRIS QUETZALI ORDOÑEZ TURCIOS,ROSALINA NADEYDA TOJ MORENTE,DIVERSIFICADO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,VESPERTINA,DIARIO(REGULAR),BAJA VERAPAZ
4,15-01-0099-46,15-01-0890,BAJA VERAPAZ,SALAMA,<NA>,COLEGIO PARTICULAR MIXTO TEZULUTLAN,"DIAGONAL 4, 1-24 ZONA 2",79400182,IRIS QUETZALI ORDOÑEZ TURCIOS,ELIA NÍVEA LÓPEZ GARCÍA,DIVERSIFICADO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,NOCTURNA,DIARIO(REGULAR),BAJA VERAPAZ


## Ejecución de las siete validaciones

In [3]:
resultados = ejecutar_validaciones(df_candidato, max_ejemplos=5)
resumen = resumen_validaciones(resultados)
resumen

,Prueba,Estado,Errores,Detalle
0,Duplicados exactos,APROBADO,0,No deben existir filas idénticas en todas las ...
1,Espacios en textos,APROBADO,0,"Los textos no deben tener espacios al inicio, ..."
2,Formato de teléfonos,APROBADO,0,"Cada teléfono debe tener 8 dígitos, iniciar en..."
3,Geografía oficial,APROBADO,0,Departamento y municipio deben pertenecer al c...
4,Esquema y tipos,APROBADO,0,El candidato debe contener las 18 columnas esp...
5,Categorías equivalentes,APROBADO,0,No deben coexistir categorías que solo difiera...
6,Valores inválidos diagnosticados,APROBADO,0,"Códigos, distritos, dominios, faltantes obliga..."


Las pruebas comprueban, en orden: duplicados exactos; espacios en textos; teléfonos; geografía oficial; esquema y tipos; categorías equivalentes por escritura; y valores inválidos identificados durante el diagnóstico.

## Detalle de fallos y ejemplos

In [4]:
fallos = [resultado for resultado in resultados if not resultado.aprobada]
if not fallos:
    print("No se encontraron fallos; las siete validaciones fueron aprobadas.")
else:
    for resultado in fallos:
        print(f"\n{resultado.prueba}: {resultado.errores} error(es)")
        print(resultado.detalle)
        print(pd.DataFrame(resultado.ejemplos).to_string(index=False))

No se encontraron fallos; las siete validaciones fueron aprobadas.


## Comprobación inequívoca del resultado

In [5]:
assert len(resultados) == 7, "Deben ejecutarse exactamente siete validaciones."
assert all(resultado.aprobada for resultado in resultados), {
    resultado.prueba: resultado.ejemplos for resultado in fallos
}
assert resumen["Errores"].sum() == 0

print("VALIDACIÓN APROBADA: 7 de 7 pruebas superadas con cero errores.")

VALIDACIÓN APROBADA: 7 de 7 pruebas superadas con cero errores.


## Esquema validado

In [6]:
pd.DataFrame(
    {
        "Variable": df_candidato.columns,
        "Tipo observado": [str(tipo) for tipo in df_candidato.dtypes],
        "Valores faltantes": [int(df_candidato[col].isna().sum()) for col in df_candidato],
        "Valores únicos": [int(df_candidato[col].nunique(dropna=True)) for col in df_candidato],
    }
)

,Variable,Tipo observado,Valores faltantes,Valores únicos
0,CODIGO,string,0,11868
1,DISTRITO,string,603,1678
2,DEPARTAMENTO,string,0,22
3,MUNICIPIO,string,0,330
4,ZONA_CAPITAL,string,9707,22
5,ESTABLECIMIENTO,string,5,6312
6,DIRECCION,string,89,7431
7,TELEFONO,string,1063,6459
8,SUPERVISOR,string,539,1279
9,DIRECTOR,string,2142,5489


## Informe reproducible de calidad Antes/Después

In [7]:
RUTA_RAW = ROOT / "data" / "raw" / "establecimientos_raw.csv"
RUTA_TRANSFORMACIONES = ROOT / "data" / "processed" / "transformaciones.csv"
RUTA_DUPLICADOS = ROOT / "data" / "processed" / "duplicados_revisados.csv"

df_raw = pd.read_csv(RUTA_RAW, dtype="string", keep_default_na=False)
transformaciones = pd.read_csv(RUTA_TRANSFORMACIONES, dtype="string")
duplicados_revisados = pd.read_csv(RUTA_DUPLICADOS, dtype="string")

comparacion = calcular_comparacion_calidad(
    df_raw, df_candidato, transformaciones
)
tabla_calidad = tabla_comparativa(comparacion)
tabla_calidad

,Métrica,Antes,Después (candidato)
0,Registros,"11,891","11,868"
1,Variables,17,18
2,Valores faltantes,"4,645 de 202,147 (2.30 %)","14,148 de 213,624 (6.62 %)"
3,Variables con NA,"17 (CODIGO, DISTRITO, DEPARTAMENTO, MUNICIPIO,...","7 (DISTRITO, ZONA_CAPITAL, ESTABLECIMIENTO, DI..."
4,Duplicados exactos,22,0
5,Posibles duplicados,"769 pares / 1,372 registros","781 pares / 1,386 registros"
6,Variables con formato inconsistente,"5 (DISTRITO, DIRECCION, TELEFONO, DIRECTOR, DE...",0 (Ninguna)
7,Variables con tipo incorrecto,0 (Ninguna),0 (Ninguna)
8,Categorías inconsistentes,16,0
9,Errores corregidos,0 (estado previo),Filas estructurales eliminadas: 23; Distritos ...


### Comparación equivalente de valores faltantes

In [8]:
pd.DataFrame(
    [
        {
            "Comparación": "Todas las variables de cada estado",
            "Antes": f"{comparacion.antes.faltantes:,} / {comparacion.antes.celdas:,} ({comparacion.antes.porcentaje_faltantes:.2f} %)",
            "Después": f"{comparacion.despues.faltantes:,} / {comparacion.despues.celdas:,} ({comparacion.despues.porcentaje_faltantes:.2f} %)",
        },
        {
            "Comparación": "Solo las 17 variables originales",
            "Antes": f"{comparacion.antes.faltantes:,} / {comparacion.antes.celdas:,} ({comparacion.antes.porcentaje_faltantes:.2f} %)",
            "Después": f"{comparacion.faltantes_despues_comparables:,} / {comparacion.celdas_despues_comparables:,} ({comparacion.porcentaje_despues_comparable:.2f} %)",
        },
    ]
)

,Comparación,Antes,Después
0,Todas las variables de cada estado,"4,645 / 202,147 (2.30 %)","14,148 / 213,624 (6.62 %)"
1,Solo las 17 variables originales,"4,645 / 202,147 (2.30 %)","4,441 / 201,756 (2.20 %)"


### Decisiones sobre posibles duplicados

In [9]:
decisiones_duplicados = resumen_decisiones_duplicados(duplicados_revisados)
pd.DataFrame(
    decisiones_duplicados.items(), columns=["Decisión", "Pares candidatos"]
)

,Decisión,Pares candidatos
0,conservados,781
1,corregidos,0
2,fusionados,0
3,eliminados,0


### Resumen de correcciones

In [10]:
pd.DataFrame(
    comparacion.correcciones.items(),
    columns=["Tipo de corrección", "Registros afectados"],
)

,Tipo de corrección,Registros afectados
0,Filas estructurales eliminadas,23
1,Distritos incompletos convertidos a NA,70
2,Teléfonos reformateados o invalidados,258
3,Valores de puntuación convertidos a NA,9
4,Prefijos decorativos corregidos,1
5,Departamentos normalizados,2161
6,Municipios normalizados,2339
7,Zonas capitalinas preservadas,2161
8,Valores de PLAN normalizados,505
9,Valores de DEPARTAMENTAL normalizados,2026


### Comprobación reproducible del informe

In [11]:
assert tabla_calidad.shape == (10, 3)
assert (comparacion.antes.registros, comparacion.despues.registros) == (11891, 11868)
assert (comparacion.antes.faltantes, comparacion.despues.faltantes) == (4645, 14148)
assert comparacion.faltantes_despues_comparables == 4441
assert (comparacion.antes.duplicados_exactos, comparacion.despues.duplicados_exactos) == (22, 0)
assert (
    comparacion.antes.posibles_duplicados_pares,
    comparacion.despues.posibles_duplicados_pares,
) == (769, 781)
assert decisiones_duplicados == {
    "conservados": 781,
    "corregidos": 0,
    "fusionados": 0,
    "eliminados": 0,
}
assert all(resultado.aprobada for resultado in resultados)

print("INFORME REPRODUCIDO: 10 métricas Antes/Después confirmadas.")

INFORME REPRODUCIDO: 10 métricas Antes/Después confirmadas.


## Resultado

El conjunto candidato supera las siete reglas automáticas y la comparación reproducible confirma las diez métricas Antes/Después. Este resultado no renombra ni exporta todavía el CSV final.